## Model selection — one honest screen
Every candidate run through the SAME protocol: train-only normalization stats,
the same first-80%-of-engines split (seed 42 in the models, not the split),
predict once at each TEST engine's last cycle vs NASA ground truth.
Headline metric: critical-zone [0-25] RMSE. Tiebreaker: NASA score. Also reported:
global RMSE, late %.
Flat models (mean, ridge, RF, xgboost) get the 42 engineered features; the LSTM gets
the 14 normalized sensor channels as 30-cycle sequences.
This is a FAMILY screen at fixed/untuned configs — see the asymmetry note in the
decision cell.

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path('../..').resolve() / 'src'))
from turbofan.evaluation.comparison import run_comparison, decision_summary

warnings.filterwarnings('ignore')
RAW = Path('../../data/raw')
MODEL_ORDER = ['mean', 'ridge', 'rf', 'xgboost', 'lstm']

In [ ]:
df = run_comparison(RAW)

In [ ]:
df['model'] = pd.Categorical(df['model'], MODEL_ORDER, ordered=True)
df = df.sort_values(['dataset', 'model'])

table = (df.set_index(['dataset', 'model'])
           [['critical_rmse', 'nasa', 'global_rmse', 'late_pct']]
           .rename(columns={'critical_rmse': '[0-25] crit RMSE', 'nasa': 'NASA score',
                            'global_rmse': 'global RMSE', 'late_pct': 'late %'})
           .round({'[0-25] crit RMSE': 2, 'global RMSE': 2, 'late %': 1, 'NASA score': 0}))
display(table)

In [ ]:
agg, wins = decision_summary(df)

print('Mean critical-zone RMSE / mean NASA across FD001-FD004 (sorted):')
print(agg.round(2))
print('\nDatasets won on critical-zone RMSE:')
print(wins)
print(f'\nLowest mean critical-zone RMSE: {agg.index[0]}')
print(f'Runner-up: {agg.index[1]}  '
      f'(margin {agg["mean_crit"].iloc[1] - agg["mean_crit"].iloc[0]:.2f} cycles)')

In [ ]:
from turbofan.evaluation.comparison import (
    run_comparison_multiseed, multiseed_summary, contender_gap,
)

# Re-run the two contenders over 5 seeds. Features + split are built once per dataset and
# reused, so only model init/training varies. (LSTM fits on FD002/FD004 dominate runtime —
# drop to seeds=(42, 7, 123) for a faster first pass.)
df_ms = run_comparison_multiseed(RAW, seeds=(42, 7, 123, 2024, 99), models=("xgboost", "lstm"))

# Banded critical-zone RMSE: mean ± std per dataset × model
display(multiseed_summary(df_ms, "critical_rmse"))

# Head-to-head: does the LSTM lead survive ±1σ on each dataset?
display(contender_gap(df_ms, a="lstm", b="xgboost"))

# Pooled across all datasets × seeds
overall = df_ms.groupby("model")["critical_rmse"].agg(["mean", "std"]).round(2)
print("Pooled critical-zone RMSE (all datasets × seeds):")
print(overall)

## Decision

**Ship the LSTM** (14 normalized sensor channels, 30-cycle sequences). Honest claim:
**best among these candidates under this protocol** — last-cycle prediction vs NASA
ground truth, headline = critical-zone [0-25] RMSE, NASA score as tiebreaker.

**Evidence (5 seeds, model init/training varied; split held fixed):**
- Pooled critical-zone RMSE: LSTM 4.65 vs XGBoost 6.92.
- LSTM separated beyond ±1σ on FD001, FD003, FD004 — on these its worst seed beats
  XGBoost's best seed (non-overlapping ranges).
- FD002 is within noise (LSTM 5.04±0.84 vs XGBoost 5.86±0.25); the single-seed XGBoost
  "win" there was a seed-42 artifact and reverses to a slight LSTM edge over five seeds.
- XGBoost is meaningfully better on no dataset.

**Why this beats the obvious objection:** the LSTM won *untuned*. XGBoost's 20-trial
per-dataset Optuna search (nb02) moved mean critical RMSE by ~0.04 cycles — it targeted
global validation RMSE, not the critical zone — so its tuning edge on the headline metric
is empirically ~zero. Tuning the LSTM would likely widen, not close, the gap.

**What was not tested / known characteristics:** train/val split composition variance
(only model seed was varied); per-model HP search beyond XGBoost's (mis-targeted) one;
one architecture per family; one RUL cap (125) and window (30). The LSTM is noisier
run-to-run than XGBoost (std up to 0.84 vs 0.42), and on FD001 carries a worse NASA score
than XGBoost despite winning critical RMSE (fatter tails outside the critical zone).